# Loss、标签与任务契约

## 学习目标

根据任务类型选择正确的输出、标签 dtype/shape、loss 和评估指标，识别广播和提前激活造成的错误。

## 概念模型

先确定任务契约：回归输出连续值；二分类通常使用 logits + `BCEWithLogitsLoss`；多分类使用 logits + `CrossEntropyLoss`；多标签分类使用独立 logits + multi-hot 标签。

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
# 回归：预测和标签都保留相同 shape，避免意外广播
regression_output = torch.randn(6, 1)
regression_target = torch.randn(6, 1)
regression_loss = nn.MSELoss()(regression_output, regression_target)
assert regression_loss.ndim == 0
try:
    nn.MSELoss()(regression_output, regression_target.squeeze(1))
except (RuntimeError, UserWarning) as error:
    print('shape warning/error:', type(error).__name__)
print('regression:', regression_loss.item())

### 实验 1：二分类和多分类损失契约

**实验目的**：对比 `BCEWithLogitsLoss` 与 `CrossEntropyLoss`。二分类 logits/float targets shape 相同；多分类 logits 为 `(batch,classes)`，targets 是 long 类别索引 `(batch,)`。

两者都接收原始 logits，不要提前 sigmoid/softmax。回归预测与目标也应保持相同 shape，避免广播产生错误损失。


In [ ]:
binary_logits = torch.randn(6)
binary_targets = torch.randint(0, 2, (6,), dtype=torch.float32)
binary_loss = nn.BCEWithLogitsLoss()(binary_logits, binary_targets)
multi_logits = torch.randn(6, 3)
multi_targets = torch.tensor([0, 1, 2, 0, 1, 2], dtype=torch.long)
multi_loss = nn.CrossEntropyLoss()(multi_logits, multi_targets)
assert binary_loss.ndim == multi_loss.ndim == 0
print('binary/multiclass:', binary_loss.item(), multi_loss.item())
try:
    nn.CrossEntropyLoss()(multi_logits, multi_targets.float())
except RuntimeError as error:
    print('expected label dtype error:', type(error).__name__)

### 实验 2：多标签输出和指标

**实验目的**：多标签任务中每个类别独立，使用 `(batch,classes)` float 0/1 targets 和 BCEWithLogitsLoss。推理对每类 sigmoid 后按阈值产生多个标签。

阈值应在验证集按业务成本选择，不必固定 0.5。多标签不能用 argmax；指标应区分 micro/macro precision、recall、F1 和每类支持度。


In [ ]:
multi_label_logits = torch.randn(4, 3)
multi_label_targets = torch.tensor([[1,0,1],[0,1,0],[1,1,0],[0,0,1]], dtype=torch.float32)
multi_label_loss = nn.BCEWithLogitsLoss()(multi_label_logits, multi_label_targets)
predictions = (multi_label_logits.sigmoid() >= 0.5)
tp = (predictions & multi_label_targets.bool()).sum(0).float()
precision = tp / predictions.sum(0).clamp_min(1)
recall = tp / multi_label_targets.sum(0).clamp_min(1)
f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-8)
print('multilabel loss/F1:', multi_label_loss.item(), f1.mean().item())
assert torch.isfinite(f1).all()

## 官方教程补充

**对应官方源文件：** `beginner_source/basics/optimization_tutorial.py`、`beginner_source/nn_tutorial.py`、`beginner_source/fgsm_tutorial.py`

官方分类示例直接把 logits 传给 `CrossEntropyLoss`，标签是 long 类别索引；二分类/多标签通常用同 shape 的浮点标签和 `BCEWithLogitsLoss`。提前 Softmax/Sigmoid 既重复计算又损害数值稳定性。预测阈值和业务指标属于评估契约，应在验证集选择；类别不平衡时同时报告按类 precision/recall/F1。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

说明四种任务的输出和标签契约；解释类别极度不平衡时 accuracy 为什么可能很高但 macro F1 很低。

## 试一试

构造全预测为多数类的例子，分别计算 accuracy、precision、recall 和 F1；比较提前 softmax 与直接传 logits 的结果。

## 常见错误与调试

`[N]` 与 `[N, 1]` 发生隐式广播、CrossEntropy 标签不是 long、BCE 标签不是 float、把多标签任务写成互斥多分类、把概率当作校准置信度。